# 🏆 Customer Lifetime Value (CLTV) Prediction System
## Phase 3: Data Cleaning & Preprocessing
**Project**: Fortune 500 E-Commerce CLTV Prediction  
**Author**: Nisarga N  
**Date**: June 2026  
**Dataset**: UCI Online Retail II  
**Objective**: Build a robust, repeatable cleaning pipeline that documents rows dropped and outlier thresholds.


In [ ]:
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to system path
sys.path.append(str(Path.cwd().parent))
from src.data_loader import DataLoader
from src.data_cleaner import DataCleaner

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'grid.color': '#21262d',
    'font.size': 11, 'axes.titlesize': 14,
})

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROCESSED = Path.cwd().parent / 'data' / 'processed'
VIZ_EDA = Path.cwd().parent / 'visualizations' / 'eda'


## 1. Load Raw Transactions
We fetch the dataset and initialize our `DataCleaner` object.


In [ ]:
loader = DataLoader(DATA_RAW)
# Load whichever file is present
raw_file = list(DATA_RAW.glob('*.csv')) + list(DATA_RAW.glob('*.xlsx'))
df_raw = loader.load_online_retail_data(raw_file[0].name)

cleaner = DataCleaner(df_raw)
print(f"Loaded raw data: {df_raw.shape[0]:,} rows")


## 2. Execute Full Cleaning Pipeline
The cleaning pipeline runs these steps:
1. Drop records where Customer ID is NULL
2. Exclude returns/cancellations (Invoice starts with 'C')
3. Exclude non-positive Quantities and Prices
4. Remove exact duplicate lines
5. Filter test/internal StockCodes (like POST, DOT, M)
6. Clip outliers (Winsorize upper 1% and lower 1% of Price/Quantity)
7. Save cleaned data with an added Revenue column


In [ ]:
# Run cleaning pipeline
df_clean = cleaner.run_full_pipeline()

# Get audit report
report = cleaner.get_cleaning_report()
report


## 3. Visualize Cleaning Audit & Outliers
Checking the cleaning summary and boxplots of Price/Quantity before and after winsorization to confirm outlier distortions are mitigated.


In [ ]:
# Outlier visualization sample
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df_clean, y="Quantity", ax=axes[0], color="#58a6ff")
axes[0].set_title("Winsorized Quantity")
sns.boxplot(data=df_clean, y="Price", ax=axes[1], color="#3fb950")
axes[1].set_title("Winsorized Price")
plt.savefig(VIZ_EDA / "07_outliers.png", dpi=150, facecolor='#0d1117')
plt.show()

# Save cleaned output
loader.save_processed(df_clean, DATA_PROCESSED / "cleaned_retail.csv")
